In [4]:
# Import required libraries
import sqlite3  
import pandas as pd  
import os  

# Set working directory
os.chdir(r'D:\Mission_Blitzkreig\Month_2_SQL')



with sqlite3.connect('marketing_campaigns.db') as conn:
   
    query = """
    SELECT 
        name,  -- Table name
        sql  -- SQL used to create the table
    FROM sqlite_master
    WHERE type='table'  -- Only show tables (not indexes or other objects)
    """
    
    df_tables = pd.read_sql_query(query, conn)
    print(" Current tables in database:")
    print(df_tables)
    
print("\n✅Connected to marketing_campaigns.db")

 Current tables in database:
                    name                                                sql
0              campaigns  CREATE TABLE "campaigns" (\n"campaign_name" TE...
1  cammpaign_perfromance  CREATE TABLE cammpaign_perfromance(\n         ...
2   campaign_performance  CREATE TABLE campaign_performance(\n    perfor...
3      city_demographics  CREATE TABLE city_demographics(\n    city_id I...

✅Connected to marketing_campaigns.db


In [5]:
#2 . Table relationships 
print(" Table Relationships in DB")

#1-1
print("One-to-One (1:1)")
print("""Definition: Each record in Table A has exactly one related record in Table B

Example: Campaign ↔ Campaign_Manager
- Campaign ID 101 → Manager ID 5 (John)
- Campaign ID 102 → Manager ID 6 (Sarah)
- Each campaign has ONE manager
- Each manager manages ONE campaign

When to use:
✓ Separating sensitive data (passwords in separate table)
✓ Optional information (not all campaigns have all attributes)
✓ Keeping tables smaller for performance"""
      )

#1-many
print("One to many(1:M)")
print("""
Definition: Each record in Table A can have multiple related records in Table B

Example: City ↔ Campaigns
- Mumbai (City ID 1) → Campaign 101, 102, 103
- Chennai (City ID 2) → Campaign 104, 105
- Each city has MANY campaigns
- Each campaign belongs to ONE city

When to use:
✓ Most common relationship type
✓ Parent-child relationships
✓ Most real-world business scenarios""")

#Many-to-Many
print("Many-to-Many(M:M)")
print("""
      Definition: Records in Table A can relate to multiple records in Table B (and vice versa)

Example: Campaigns ↔ Platforms (requires junction table!)
- Campaign 101 → Facebook, Instagram, Twitter
- Campaign 102 → Instagram, TikTok
- Campaign 103 → Facebook, TikTok
- Campaigns use MANY platforms
- Platforms are used by MANY campaigns

Tables needed:
1. campaigns (Campaign ID, Name)
2. platforms (Platform ID, Name)
3. campaign_platforms (Campaign ID, Platform ID) ← JUNCTION TABLE

When to use:
✓ When two tables need multiple connections
✓ Requires a junction/bridge table
✓ Breaks down into two 1:M relationships""")
      

 Table Relationships in DB
One-to-One (1:1)
Definition: Each record in Table A has exactly one related record in Table B

Example: Campaign ↔ Campaign_Manager
- Campaign ID 101 → Manager ID 5 (John)
- Campaign ID 102 → Manager ID 6 (Sarah)
- Each campaign has ONE manager
- Each manager manages ONE campaign

When to use:
✓ Separating sensitive data (passwords in separate table)
✓ Optional information (not all campaigns have all attributes)
✓ Keeping tables smaller for performance
One to many(1:M)

Definition: Each record in Table A can have multiple related records in Table B

Example: City ↔ Campaigns
- Mumbai (City ID 1) → Campaign 101, 102, 103
- Chennai (City ID 2) → Campaign 104, 105
- Each city has MANY campaigns
- Each campaign belongs to ONE city

When to use:
✓ Most common relationship type
✓ Parent-child relationships
✓ Most real-world business scenarios
Many-to-Many(M:M)

      Definition: Records in Table A can relate to multiple records in Table B (and vice versa)

Example:

In [6]:
# 3. Primary & foriegn Keys
print("Primary and Foriegn Keys")
print("PRIMARY KEYS")

print("""Definition: A column (or columns) that UNIQUELY identifies each row

Rules:
✓ Must be UNIQUE (no two rows have same value)
✓ Cannot be NULL (must have a value)
✓ One per table
✓ Usually an ID number

Example:
campaigns table:
│ campaign_id (PK) │ campaign_name  │ city      │ spend  │
├──────────────────┼────────────────┼───────────┼────────┤
│ 1                │ Bus Stop Ad     │ Mumbai    │ 5000   │
│ 2                │ Billboard       │ Chennai   │ 6200   │
│ 3                │ Transit Ad      │ Delhi     │ 4500   │
                ↑
          Primary Key (unique)""")

print("SECONDARY KEY")
print("""Definition: A column in one table that references the PRIMARY KEY of another table

Purpose:
✓ Links tables together
✓ Ensures referential integrity (no orphan records)
✓ Maintains data consistency

Example:
campaigns table:
│ campaign_id │ campaign_name  │ manager_id (FK) │ spend  │
├─────────────┼────────────────┼─────────────────┼────────┤
│ 1           │ Bus Stop Ad    │ 5               │ 5000   │
│ 2           │ Billboard      │ 6               │ 6200   │
│ 3           │ Transit Ad     │ 5               │ 4500   │
                                 ↑
                          Foreign Key (references managers.manager_id)

managers table:
│ manager_id (PK) │ manager_name │ email          │
├─────────────────┼──────────────┼────────────────┤
│ 5               │ John         │ john@gmail.com │
│ 6               │ Sarah        │ sarah@...      │
        ↑
   Primary Key""")

Primary and Foriegn Keys
PRIMARY KEYS
Definition: A column (or columns) that UNIQUELY identifies each row

Rules:
✓ Must be UNIQUE (no two rows have same value)
✓ Cannot be NULL (must have a value)
✓ One per table
✓ Usually an ID number

Example:
campaigns table:
│ campaign_id (PK) │ campaign_name  │ city      │ spend  │
├──────────────────┼────────────────┼───────────┼────────┤
│ 1                │ Bus Stop Ad     │ Mumbai    │ 5000   │
│ 2                │ Billboard       │ Chennai   │ 6200   │
│ 3                │ Transit Ad      │ Delhi     │ 4500   │
                ↑
          Primary Key (unique)
SECONDARY KEY
Definition: A column in one table that references the PRIMARY KEY of another table

Purpose:
✓ Links tables together
✓ Ensures referential integrity (no orphan records)
✓ Maintains data consistency

Example:
campaigns table:
│ campaign_id │ campaign_name  │ manager_id (FK) │ spend  │
├─────────────┼────────────────┼─────────────────┼────────┤
│ 1           │ Bus Stop Ad   

In [9]:
#4. BUILD A BETTER DB with Multiple Tables : 
print("Create a normalised DB:")

schema_db = sqlite3.connect('marketing_campaigns_normalized.db')
schema_cursor = schema_db.cursor()

#Table 1 : CITIES 
create_cities = """ 
CREATE TABLE IF NOT EXISTS cities (
      city_id INTEGER PRIMARY KEY AUTOINCREMENT,
      city_name TEXT NOT NULL UNIQUE,
      region TEXT,
      created_at DATETIME DEFAULT CURRENT_TIMESTAMP)
      """
schema_cursor.execute(create_cities)
print("cities table created")

# Table 2 : Campaign Managers 
create_managers = """
CREATE TABLE IF NOT EXISTS campaign_managers (
    manager_id INTEGER PRIMARY KEY AUTOINCREMENT,  -- Unique identifier
    manager_name TEXT NOT NULL,  -- Manager name
    email TEXT NOT NULL UNIQUE,  -- Email (unique)
    phone TEXT,  -- Phone number
    created_at DATETIME DEFAULT CURRENT_TIMESTAMP  -- When hired
)
"""
schema_cursor.execute(create_managers)
print("campaign_managers table created")

# Table 3 : Campaign Format 
create_formats = """
CREATE TABLE IF NOT EXISTS campaign_formats (
    format_id INTEGER PRIMARY KEY AUTOINCREMENT,  -- Unique identifier
    format_name TEXT NOT NULL UNIQUE,  -- Format type (Bus, Billboard, etc.)
    description TEXT,  -- Description of format
    average_cpm DECIMAL(10, 2)  -- Average CPM for this format
)
"""
schema_cursor.execute(create_formats)
print("✅ campaign_formats table created")

# Table 4 : Campaigns 
create_campaigns = """
CREATE TABLE IF NOT EXISTS campaigns (
    campaign_id INTEGER PRIMARY KEY AUTOINCREMENT,  -- Unique identifier
    campaign_name TEXT NOT NULL,  -- Campaign name
    city_id INTEGER NOT NULL,  -- Foreign key to cities table
    manager_id INTEGER NOT NULL,  -- Foreign key to managers
    format_id INTEGER NOT NULL,  -- Foreign key to formats
    start_date DATE NOT NULL,  -- Campaign start date
    end_date DATE NOT NULL,  -- Campaign end date
    budget DECIMAL(10, 2) NOT NULL,  -- Total budget
    actual_spend DECIMAL(10, 2),  -- What was actually spent
    clicks INTEGER DEFAULT 0,  -- Number of clicks
    conversions INTEGER DEFAULT 0,  -- Number of conversions
    created_at DATETIME DEFAULT CURRENT_TIMESTAMP,  -- When record created
    
    -- Define FOREIGN KEYS (links to other tables)
    FOREIGN KEY (city_id) REFERENCES cities(city_id),  -- Links to cities
    FOREIGN KEY (manager_id) REFERENCES campaign_managers(manager_id),  -- Links to managers
    FOREIGN KEY (format_id) REFERENCES campaign_formats(format_id)  -- Links to formats
)
"""

schema_cursor.execute(create_campaigns)
print("campaigns table created")

# Table 5 : Daily_performance 
create_daily = """
CREATE TABLE IF NOT EXISTS daily_performance (
    performance_id INTEGER PRIMARY KEY AUTOINCREMENT,  -- Unique identifier
    campaign_id INTEGER NOT NULL,  -- Which campaign
    performance_date DATE NOT NULL,  -- What day
    daily_spend DECIMAL(10, 2),  -- Spend that day
    daily_clicks INTEGER DEFAULT 0,  -- Clicks that day
    daily_conversions INTEGER DEFAULT 0,  -- Conversions that day
    daily_impressions INTEGER DEFAULT 0,  -- Impressions that day
    
    FOREIGN KEY (campaign_id) REFERENCES campaigns(campaign_id)  -- Links to campaigns
)
"""
schema_cursor.execute(create_daily)
print("daily_performance table created")
schema_db.commit()
print("\n✅ All tables created successfully!")
print("✅ Database file: marketing_campaigns_normalized.db")

schema_db.close()
print("✅ Connection closed")



Create a normalised DB:
cities table created
campaign_managers table created
✅ campaign_formats table created
campaigns table created
daily_performance table created

✅ All tables created successfully!
✅ Database file: marketing_campaigns_normalized.db
✅ Connection closed


In [18]:
print("\n" + "="*60)
print("INSERTING SAMPLE DATA")
print("="*60)

# Open the normalized database
with sqlite3.connect('marketing_campaigns_normalized.db') as conn:
    cursor = conn.cursor()
    
    # INSERT CITIES
    print("\n📍 Inserting cities...")
    cities_data = [
        ('Mumbai', 'West'),
        ('Chennai', 'South'),
        ('Delhi', 'North'),
        ('Bangalore', 'South'),
        ('Pune', 'West'),
        ('Goa', 'West')
    ]
    
    cursor.executemany(
        "INSERT INTO cities (city_name, region) VALUES (?, ?)",  
        cities_data  
    )
    print(f"✅ Inserted {len(cities_data)} cities")
    
    # INSERT MANAGERS
    print("📋 Inserting managers...")
    managers_data = [
        ('Ashwin', 'ashwin@company.com', '9876543210'), 
        ('Sarah', 'sarah@company.com', '9876543211'),
        ('John', 'john@company.com', '9876543212'),
        ('Priya', 'priya@company.com', '9876543213')
    ]
    
    cursor.executemany(
        "INSERT INTO campaign_managers (manager_name, email, phone) VALUES (?, ?, ?)",
        managers_data
    )
    print(f"✅ Inserted {len(managers_data)} managers")
    
    # INSERT FORMATS
    print("🎨 Inserting campaign formats...")
    formats_data = [
        ('Bus', 'Advertisements on buses', 150.00),  
        ('Billboard', 'Large billboards', 200.00),
        ('Transit', 'Transit station ads', 175.00),
        ('Digital', 'Digital displays', 120.00)
    ]
    
    cursor.executemany(
        "INSERT INTO campaign_formats (format_name, description, average_cpm) VALUES (?, ?, ?)",
        formats_data
    )
    print(f"✅ Inserted {len(formats_data)} formats")
    
    # INSERT CAMPAIGNS (using foreign keys)
    print("📊 Inserting campaigns...")
    campaigns_data = [
        ('Summer Bus Campaign', 1, 1, 1, '2024-01-01', '2024-02-01', 50000, 45000, 2500, 150),
        ('Billboard Blitz', 2, 2, 2, '2024-01-15', '2024-03-15', 60000, 58000, 3200, 200),
        ('Digital Push', 1, 3, 4, '2024-02-01', '2024-02-28', 35000, 34000, 1800, 90),
        ('Transit Network', 3, 4, 3, '2024-01-20', '2024-03-20', 55000, 52000, 2800, 175)
    ]
    
    cursor.executemany(
        """INSERT INTO campaigns 
        (campaign_name, city_id, manager_id, format_id, start_date, end_date, budget, actual_spend, clicks, conversions)
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)""",
        campaigns_data
    )
    print(f"✅ Inserted {len(campaigns_data)} campaigns")
    
    # Commit all changes
    conn.commit()
    
    print("\n✅ All data inserted successfully!")


INSERTING SAMPLE DATA

📍 Inserting cities...


IntegrityError: UNIQUE constraint failed: cities.city_name

In [21]:
# 6. Show how relationships work:
print("QUERYING THE NORMALIZED DATABASE")
with sqlite3.connect('marketing_campaigns_normalized.db') as conn:

    print("\n CAMPAIGNS WITH FULL DETAILS (using JOINs):")

    query1 = """
SELECT 
        c.campaign_name,  -- Campaign name
        ci.city_name,  -- City (from cities table via FK)
        cm.manager_name,  -- Manager name (from managers table via FK)
        cf.format_name,  -- Format (from formats table via FK)
        c.actual_spend,  -- Actual spend
        c.conversions  -- Conversions
    FROM campaigns c  -- Main table
    INNER JOIN cities ci ON c.city_id = ci.city_id  -- Join cities
    INNER JOIN campaign_managers cm ON c.manager_id = cm.manager_id  -- Join managers
    INNER JOIN campaign_formats cf ON c.format_id = cf.format_id  -- Join formats
    ORDER BY c.campaign_name
    """
    
    df1 = pd.read_sql_query(query1, conn)
    print(df1.to_string(index=False))

    #query 2 : Summary by city 
print("\nSUMMARY BY CITY:")

query2 = """
    SELECT 
        ci.city_name,  -- City name
        ci.region,  -- Region
        COUNT(c.campaign_id) as campaign_count,  -- How many campaigns
        SUM(c.actual_spend) as total_spend,  -- Total spend in this city
        AVG(c.conversions) as avg_conversions  -- Average conversions
    FROM cities ci
    LEFT JOIN campaigns c ON ci.city_id = c.city_id  -- Join campaigns
    GROUP BY ci.city_id, ci.city_name, ci.region
    ORDER BY total_spend DESC
    """
    
df2 = pd.read_sql_query(query2, conn)
print(df2.to_string(index=False))
    
    # QUERY 3: Manager performance (who manages most campaigns)
print("\n\n3️⃣  MANAGER PERFORMANCE:")
print("-" * 60)
    
query3 = """
    SELECT 
        cm.manager_name,  -- Manager name
        COUNT(c.campaign_id) as campaigns_managed,  -- Number of campaigns
        SUM(c.actual_spend) as total_budget_managed,  -- Total budget managed
        AVG(c.conversions) as avg_conversions  -- Average conversions
    FROM campaign_managers cm
    LEFT JOIN campaigns c ON cm.manager_id = c.manager_id
    GROUP BY cm.manager_id, cm.manager_name
    ORDER BY campaigns_managed DESC
    """
    
df3 = pd.read_sql_query(query3, conn)
print(df3.to_string(index=False))
    
    # QUERY 4: Format performance
print("\n\n4️⃣  FORMAT PERFORMANCE:")
print("-" * 60)
    
query4 = """
    SELECT 
        cf.format_name,  -- Format name
        cf.average_cpm,  -- Average CPM for format
        COUNT(c.campaign_id) as usage_count,  -- How many times used
        AVG(c.conversions) as avg_conversions,  -- Average conversions
        AVG(c.clicks) as avg_clicks  -- Average clicks
    FROM campaign_formats cf
    LEFT JOIN campaigns c ON cf.format_id = c.format_id
    GROUP BY cf.format_id, cf.format_name
    ORDER BY usage_count DESC
    """
    
df4 = pd.read_sql_query(query4, conn)
print(df4.to_string(index=False))

print("\n✅ All queries executed successfully!")



QUERYING THE NORMALIZED DATABASE

 CAMPAIGNS WITH FULL DETAILS (using JOINs):
      campaign_name city_name manager_name format_name  actual_spend  conversions
    Billboard Blitz   Chennai        Sarah   Billboard         58000          200
       Digital Push    Mumbai         John     Digital         34000           90
Summer Bus Campaign    Mumbai       Ashwin         Bus         45000          150
    Transit Network     Delhi        Priya     Transit         52000          175

SUMMARY BY CITY:
city_name region  campaign_count  total_spend  avg_conversions
   Mumbai   West               2      79000.0            120.0
  Chennai  South               1      58000.0            200.0
    Delhi  North               1      52000.0            175.0
Bangalore  South               0          NaN              NaN
      Goa   West               0          NaN              NaN
     Pune   West               0          NaN              NaN


3️⃣  MANAGER PERFORMANCE:
-------------------------

In [23]:
print("\n" + "="*60)
print("NORMALIZATION PRINCIPLES")
print("="*60)

print("""
NORMALIZATION: Process of organizing data to reduce redundancy and improve integrity

🎯 Three Main Normal Forms:

1️⃣  FIRST NORMAL FORM (1NF)
   Requirement: Each column contains only atomic (indivisible) values
   
   ❌ BAD (violates 1NF):
   campaigns table:
   │ campaign_id │ campaign_name │ cities              │ formats           │
   ├─────────────┼───────────────┼─────────────────────┼───────────────────┤
   │ 1           │ Summer Ad     │ Mumbai, Delhi       │ Bus, Billboard    │
                                   (multiple values!)    (multiple values!)
   
   ✅ GOOD (follows 1NF):
   campaigns table:
   │ campaign_id │ campaign_name │ city_id │ format_id │
   ├─────────────┼───────────────┼─────────┼───────────┤
   │ 1           │ Summer Ad     │ 1       │ 1         │
   │ 1           │ Summer Ad     │ 3       │ 2         │
                   (separate rows for combinations)

---

2️⃣  SECOND NORMAL FORM (2NF)
   Requirement: 1NF + Remove partial dependencies
   (Non-key columns must depend on ENTIRE primary key, not part of it)
   
   ❌ BAD (violates 2NF):
   campaign_details table:
   │ campaign_id │ manager_id │ manager_email    │
   ├─────────────┼────────────┼──────────────────┤
   │ 1           │ 5          │ john@company.com │
                   (manager_email depends only on manager_id, not campaign_id!)
   
   ✅ GOOD (follows 2NF):
   campaigns table:
   │ campaign_id │ manager_id │
   ├─────────────┼────────────┤
   │ 1           │ 5          │
   
   managers table (separate):
   │ manager_id │ manager_email    │
   ├────────────┼──────────────────┤
   │ 5          │ john@company.com │

---

3️⃣  THIRD NORMAL FORM (3NF)
   Requirement: 2NF + Remove transitive dependencies
   (Non-key columns should not depend on other non-key columns)
   
   ❌ BAD (violates 3NF):
   campaigns table:
   │ campaign_id │ city_id │ city_name │ region    │
   ├─────────────┼─────────┼───────────┼───────────┤
   │ 1           │ 1       │ Mumbai    │ West      │
                            (region depends on city_name, not campaign_id!)
   
   ✅ GOOD (follows 3NF):
   campaigns table:
   │ campaign_id │ city_id │
   ├─────────────┼─────────┤
   │ 1           │ 1       │
   
   cities table (separate):
   │ city_id │ city_name │ region │
   ├─────────┼───────────┼────────┤
   │ 1       │ Mumbai    │ West   │

---

BENEFITS OF NORMALIZATION:
✅ Reduces data redundancy (no duplicate data)
✅ Improves data integrity (single source of truth)
✅ Faster updates (change once, applies everywhere)
✅ More flexible queries (JOINs for different perspectives)
✅ Easier maintenance (clear structure)
""")

print("\n" + "="*60)




NORMALIZATION PRINCIPLES

NORMALIZATION: Process of organizing data to reduce redundancy and improve integrity

🎯 Three Main Normal Forms:

1️⃣  FIRST NORMAL FORM (1NF)
   Requirement: Each column contains only atomic (indivisible) values

   ❌ BAD (violates 1NF):
   campaigns table:
   │ campaign_id │ campaign_name │ cities              │ formats           │
   ├─────────────┼───────────────┼─────────────────────┼───────────────────┤
   │ 1           │ Summer Ad     │ Mumbai, Delhi       │ Bus, Billboard    │
                                   (multiple values!)    (multiple values!)

   ✅ GOOD (follows 1NF):
   campaigns table:
   │ campaign_id │ campaign_name │ city_id │ format_id │
   ├─────────────┼───────────────┼─────────┼───────────┤
   │ 1           │ Summer Ad     │ 1       │ 1         │
   │ 1           │ Summer Ad     │ 3       │ 2         │
                   (separate rows for combinations)

---

2️⃣  SECOND NORMAL FORM (2NF)
   Requirement: 1NF + Remove partial dependen

In [25]:
#8.Ensure data consistency:

print("\n" + "="*60)
print("REFERENTIAL INTEGRITY & CONSTRAINTS")
print("="*60)

print("""
REFERENTIAL INTEGRITY: Rule that foreign key values must match primary key values

Example Problem:
campaigns table:
│ campaign_id │ manager_id │ campaign_name    │
├─────────────┼────────────┼──────────────────┤
│ 1           │ 5          │ Summer Campaign  │
│ 2           │ 10         │ Winter Campaign  │ ← manager_id 10 doesn't exist!
                              (ORPHAN RECORD - Referential Integrity Violation!)

managers table:
│ manager_id │ manager_name │
├────────────┼──────────────┤
│ 5          │ John         │
│ 6          │ Sarah        │
            (No manager_id 10!)

---

SOLUTION: Use FOREIGN KEY constraints with CASCADE actions

Options:
1. RESTRICT (default)
   - Don't allow deletion if foreign key references it
   - Error: "Cannot delete manager 5, campaigns depend on it"
   
2. CASCADE
   - Delete related records automatically
   - Delete manager 5 → automatically delete all campaigns by manager 5
   
3. SET NULL
   - Set foreign key to NULL when parent is deleted
   - Delete manager 5 → campaigns.manager_id becomes NULL

Example with CASCADE:
When you delete a manager → automatically delete all their campaigns
This prevents orphan records!
""")

print("\n" + "="*60)


REFERENTIAL INTEGRITY & CONSTRAINTS

REFERENTIAL INTEGRITY: Rule that foreign key values must match primary key values

Example Problem:
campaigns table:
│ campaign_id │ manager_id │ campaign_name    │
├─────────────┼────────────┼──────────────────┤
│ 1           │ 5          │ Summer Campaign  │
│ 2           │ 10         │ Winter Campaign  │ ← manager_id 10 doesn't exist!
                              (ORPHAN RECORD - Referential Integrity Violation!)

managers table:
│ manager_id │ manager_name │
├────────────┼──────────────┤
│ 5          │ John         │
│ 6          │ Sarah        │
            (No manager_id 10!)

---

SOLUTION: Use FOREIGN KEY constraints with CASCADE actions

Options:
1. RESTRICT (default)
   - Don't allow deletion if foreign key references it
   - Error: "Cannot delete manager 5, campaigns depend on it"

2. CASCADE
   - Delete related records automatically
   - Delete manager 5 → automatically delete all campaigns by manager 5

3. SET NULL
   - Set foreign ke

In [27]:
#9.Best Practices for Marketing Databases
print("\n" + "="*60)
print("BEST PRACTICES FOR MARKETING DATABASES")
print("="*60)

best_practices = """
1️⃣  USE CLEAR NAMING CONVENTIONS
   ✅ Tables: plural, snake_case → campaigns, campaign_managers, daily_performance
   ✅ Columns: snake_case, descriptive → campaign_id, manager_name, actual_spend
   ✅ Foreign keys: table_name_id → campaign_id, manager_id, city_id
   ✅ Booleans: is_active, has_content, should_notify

2️⃣  ALWAYS USE PRIMARY KEYS
   ✅ Every table needs a unique identifier
   ✅ Use ID columns (campaign_id, manager_id, city_id)
   ✅ Makes updates and deletes unambiguous

3️⃣  USE FOREIGN KEYS
   ✅ Links tables together safely
   ✅ Prevents orphan records
   ✅ Makes relationships explicit and enforceable

4️⃣  NORMALIZE YOUR DATA (but not too much!)
   ✅ Reduce redundancy
   ✅ But don't create so many tables it's hard to query
   ✅ Balance between normalization and query simplicity

5️⃣  ADD AUDIT COLUMNS
   ✅ created_at → When record was created
   ✅ updated_at → When record was last updated
   ✅ created_by → Who created the record
   ✅ updated_by → Who last updated it

6️⃣  CHOOSE APPROPRIATE DATA TYPES
   ✅ campaign_id: INTEGER (for IDs)
   ✅ campaign_name: TEXT (for strings)
   ✅ spend: DECIMAL(10,2) (for money - never use FLOAT for money!)
   ✅ created_at: DATETIME (for timestamps)
   ✅ is_active: BOOLEAN (for yes/no)

7️⃣  INDEX IMPORTANT COLUMNS
   ✅ Index columns used in WHERE clauses
   ✅ Index columns used in JOINs
   ✅ Index date columns for time-series queries
   ✅ Too many indexes slow down inserts, too few slow down queries

8️⃣  DOCUMENT YOUR SCHEMA
   ✅ Comments in table creation explaining purpose
   ✅ Data dictionary (what each column means)
   ✅ Business rules (constraints, relationships)
   ✅ Example queries for common operations

9️⃣  USE TRANSACTIONS
   ✅ Group related updates together
   ✅ If anything fails, rollback everything
   ✅ Prevents partial updates that corrupt data

🔟 MONITOR GROWTH
   ✅ Archive old data periodically
   ✅ Monitor database size
   ✅ Plan for growth (will data exceed 1GB? 1TB?)
"""

print(best_practices)
print("\n" + "="*60)


BEST PRACTICES FOR MARKETING DATABASES

1️⃣  USE CLEAR NAMING CONVENTIONS
   ✅ Tables: plural, snake_case → campaigns, campaign_managers, daily_performance
   ✅ Columns: snake_case, descriptive → campaign_id, manager_name, actual_spend
   ✅ Foreign keys: table_name_id → campaign_id, manager_id, city_id
   ✅ Booleans: is_active, has_content, should_notify

2️⃣  ALWAYS USE PRIMARY KEYS
   ✅ Every table needs a unique identifier
   ✅ Use ID columns (campaign_id, manager_id, city_id)
   ✅ Makes updates and deletes unambiguous

3️⃣  USE FOREIGN KEYS
   ✅ Links tables together safely
   ✅ Prevents orphan records
   ✅ Makes relationships explicit and enforceable

4️⃣  NORMALIZE YOUR DATA (but not too much!)
   ✅ Reduce redundancy
   ✅ But don't create so many tables it's hard to query
   ✅ Balance between normalization and query simplicity

5️⃣  ADD AUDIT COLUMNS
   ✅ created_at → When record was created
   ✅ updated_at → When record was last updated
   ✅ created_by → Who created the record


In [29]:
#10.Practice - Design a Schema
print("\n" + "="*60)
print("PRACTICE: DESIGN A SCHEMA FOR A REAL-WORLD SCENARIO")
print("="*60)

scenario = """
SCENARIO: E-Commerce Platform for Digital Marketing Services

Requirements:
- Users (customers) who book campaigns
- Campaigns with multiple platforms (Facebook, Instagram, TikTok, etc.)
- Platforms have different pricing and capabilities
- Each user can create multiple campaigns
- Each campaign targets multiple platforms
- Track daily performance metrics for each campaign-platform combination
- Track invoices and payments for users

QUESTIONS:
1. How many tables do you need?
2. What are the primary keys?
3. What are the foreign keys?
4. What are the one-to-many relationships?
5. What is the many-to-many relationship?

---

SOLUTION:

Tables needed:
1. users → user_id (PK), email, company_name, created_at
2. campaigns → campaign_id (PK), user_id (FK), campaign_name, budget, status
3. platforms → platform_id (PK), platform_name, description, cost_per_impression
4. campaign_platforms → campaign_id (FK), platform_id (FK) ← JUNCTION TABLE
5. daily_metrics → metric_id (PK), campaign_id (FK), platform_id (FK), date, clicks, impressions
6. invoices → invoice_id (PK), user_id (FK), amount, date
7. invoice_items → item_id (PK), invoice_id (FK), campaign_id (FK), amount

Relationships:
- users 1:M campaigns (one user has many campaigns)
- campaigns M:M platforms (many campaigns on many platforms)
- campaigns 1:M daily_metrics (one campaign has many daily records)
- users 1:M invoices (one user has many invoices)
- invoices 1:M invoice_items (one invoice has many line items)
"""

print(scenario)
print("\n" + "="*60)


PRACTICE: DESIGN A SCHEMA FOR A REAL-WORLD SCENARIO

SCENARIO: E-Commerce Platform for Digital Marketing Services

Requirements:
- Users (customers) who book campaigns
- Campaigns with multiple platforms (Facebook, Instagram, TikTok, etc.)
- Platforms have different pricing and capabilities
- Each user can create multiple campaigns
- Each campaign targets multiple platforms
- Track daily performance metrics for each campaign-platform combination
- Track invoices and payments for users

QUESTIONS:
1. How many tables do you need?
2. What are the primary keys?
3. What are the foreign keys?
4. What are the one-to-many relationships?
5. What is the many-to-many relationship?

---

SOLUTION:

Tables needed:
1. users → user_id (PK), email, company_name, created_at
2. campaigns → campaign_id (PK), user_id (FK), campaign_name, budget, status
3. platforms → platform_id (PK), platform_name, description, cost_per_impression
4. campaign_platforms → campaign_id (FK), platform_id (FK) ← JUNCTION TAB

In [30]:
print("\n✅ Day 28: Database Design completed!")
print("✅ You now understand:")
print("   - Table relationships (1:1, 1:M, M:M)")
print("   - Primary and foreign keys")
print("   - Normalization principles")
print("   - Best practices for production databases")


✅ Day 28: Database Design completed!
✅ You now understand:
   - Table relationships (1:1, 1:M, M:M)
   - Primary and foreign keys
   - Normalization principles
   - Best practices for production databases
